# 📊 Telecom Churn Analysis using SageMaker Canvas

## Overview
This project builds a churn prediction pipeline using Amazon SageMaker Canvas.  
It includes data cleaning, feature engineering, class balancing (SMOTE), and model training.

Some steps were performed using the Canvas visual interface, while feature engineering was done using PySpark.

## 📊 Pipeline Overview

This workflow demonstrates an end-to-end machine learning pipeline built in Amazon SageMaker Canvas:

1. **Data Preparation**: Load and explore customer churn data
2. **Data Cleaning**: Remove missing values, handle duplicates, and correct data types
3. **Class Balancing**: Apply SMOTE to balance imbalanced churn classes
4. **Feature Engineering**: Create meaningful features using PySpark
5. **Exploratory Analysis**: Understand patterns and relationships in the data
6. **Model Training**: Use SageMaker Canvas AutoML to automatically select the best model
7. **Evaluation**: Assess model performance using relevant metrics

## 🧹 Data Cleaning

### Key Data Cleaning Steps:

- **Removed missing values**: Identified and handled null/NaN entries across all columns
- **Filtered invalid rows**: Removed records with inconsistent or impossible values
- **Data types corrected**: Ensured appropriate data types (numeric, categorical, datetime)
- **Duplicates handled**: Identified and removed duplicate customer records

These steps were performed using the SageMaker Canvas visual interface to ensure data quality before feature engineering.

## ⚖️ Handling Class Imbalance

### Problem:
The churn dataset was highly imbalanced with significantly fewer churned customers than retained customers.

### Solution:
Applied **SMOTE (Synthetic Minority Over-sampling Technique)** to:
- Generate synthetic samples for the minority class (churned customers)
- Balance the class distribution
- Improve model performance and reduce bias toward the majority class

This technique helps the model learn better representations of both classes.

## ⚙️ Feature Engineering

Feature engineering was performed using PySpark to create meaningful predictive features:

In [ ]:
from pyspark.sql.functions import when, col, sum as spark_sum

# Feature 1: Tenure Group - Categorize customers by tenure length
df = df.withColumn(
    'TenureGroup',
    when(col('tenure') < 12, '0-1 Year')
    .when(col('tenure') < 24, '1-2 Years')
    .when(col('tenure') < 48, '2-4 Years')
    .otherwise('4+ Years')
)

# Feature 2: Total Services - Count the number of services subscribed
service_columns = ['PhoneService', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df = df.withColumn('TotalServices', sum(col(c).cast('int') for c in service_columns))

# Feature 3: Contract Risk Level - Based on contract type and monthly charges
df = df.withColumn(
    'ContractRiskLevel',
    when((col('contract') == 'Month-to-month') & (col('MonthlyCharges') > 50), 'HighRisk')
    .when((col('contract') == 'Month-to-month') | (col('MonthlyCharges') > 75), 'MediumRisk')
    .otherwise('LowRisk')
)

# Feature 4: Lifetime Spend - Calculate total revenue per customer
df = df.withColumn('LifetimeSpend', col('tenure') * col('MonthlyCharges'))

# Display engineered features
df.select('TenureGroup', 'TotalServices', 'ContractRiskLevel', 'LifetimeSpend').show()

## 📊 Exploratory Data Analysis

### Key Insights:

- **Tenure Impact**: Customers with low tenure (less than 12 months) have significantly higher churn rates
- **Monthly Charges**: Higher monthly charges correlate with increased churn probability
- **Contract Type**: Customers with month-to-month contracts show higher churn compared to long-term contracts
- **Service Adoption**: Customers with more services tend to have lower churn rates
- **Internet Service**: Fiber optic customers have higher churn compared to DSL users

These patterns guided feature selection and model development strategy.

## 🤖 Model Training

### SageMaker Canvas AutoML:

- **Model Training**: Built using SageMaker Canvas visual interface for automatic model selection
- **Algorithm Selection**: Canvas tested multiple algorithms and automatically selected the best performer
- **Hyperparameter Tuning**: Canvas optimized hyperparameters for best validation performance
- **Model Evaluation**: Evaluated using key metrics:
  - **Accuracy**: Overall correctness of predictions
  - **AUC (Area Under the ROC Curve)**: Model's ability to distinguish between classes
  - **Precision & Recall**: Balance between false positives and false negatives

## ✅ Conclusion

This project successfully demonstrates:

✅ **End-to-End Pipeline**: Built a complete churn prediction workflow from data cleaning to model deployment

✅ **SageMaker Canvas Integration**: Leveraged AWS SageMaker Canvas for visual ML model development without extensive coding

✅ **Feature Engineering**: Implemented domain-driven feature engineering using PySpark to improve model interpretability and performance

✅ **Class Balancing**: Applied SMOTE to handle imbalanced data and improve model robustness

### Business Impact:
- Early identification of high-risk customers
- Targeted retention strategies for at-risk segments
- Improved customer lifetime value through proactive churn prevention